# Model Inference — IEEE-CIS Fraud Detection
Loads best model from MLflow Model Registry, runs on test set, generates Kaggle submission.

## 0. Setup

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'mlflow', 'dagshub', '--quiet'], capture_output=True)

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import dagshub

print('Setup complete!')

In [ ]:
# ================================================================
# Connect to DagsHub / MLflow
# ================================================================
DAGSHUB_USERNAME = 'YOUR_DAGSHUB_USERNAME'
DAGSHUB_REPO     = 'YOUR_REPO_NAME'

dagshub.init(repo_owner=DAGSHUB_USERNAME, repo_name=DAGSHUB_REPO, mlflow=True)
print(f'MLflow URI: {mlflow.get_tracking_uri()}')

## 1. Load Test Data

In [ ]:
BASE = '/kaggle/input/ieee-fraud-detection/'

test_transaction = pd.read_csv(BASE + 'test_transaction.csv')
test_identity    = pd.read_csv(BASE + 'test_identity.csv')
test = test_transaction.merge(test_identity, on='TransactionID', how='left')

print(f'Test shape: {test.shape}')
print('Columns sample:', test.columns[:10].tolist())

## 2. Load Best Model from Registry

In [ ]:
# ================================================================
# Compare all models — pick best by OOF AUC
# ================================================================
client = mlflow.tracking.MlflowClient()

model_names = [
    'XGBoost_Fraud_Pipeline',
    'LightGBM_Fraud_Pipeline',
    'CatBoost_Fraud_Pipeline',
    'RandomForest_Fraud_Pipeline',
    'LogisticRegression_Fraud_Pipeline',
]

model_scores = {}
for model_name in model_names:
    try:
        versions = client.get_latest_versions(model_name)
        if versions:
            run_id = versions[0].run_id
            run    = client.get_run(run_id)
            oof_auc = run.data.metrics.get('oof_auc', 0)
            model_scores[model_name] = oof_auc
            print(f'{model_name}: OOF AUC = {oof_auc:.4f}')
    except Exception as e:
        print(f'{model_name}: not found ({e})')

best_model_name = max(model_scores, key=model_scores.get)
print(f'\nBest model: {best_model_name} (AUC = {model_scores[best_model_name]:.4f})')

In [ ]:
# ================================================================
# Load the best model pipeline from Model Registry
# ================================================================
model_uri = f'models:/{best_model_name}/latest'
pipeline  = mlflow.sklearn.load_model(model_uri)

print(f'Loaded pipeline: {pipeline}')
print(f'Pipeline steps: {[step[0] for step in pipeline.steps]}')

## 3. Generate Predictions

In [ ]:
# Drop columns the pipeline doesn't need
X_test_raw = test.drop(columns=['TransactionID', 'TransactionDT'], errors='ignore')

# Pipeline handles all preprocessing internally
test_preds = pipeline.predict_proba(X_test_raw)[:, 1]

print(f'Predictions shape: {test_preds.shape}')
print(f'Prediction range: [{test_preds.min():.4f}, {test_preds.max():.4f}]')
print(f'Mean prediction: {test_preds.mean():.4f}')

## 4. (Optional) Ensemble All Models

In [ ]:
# ================================================================
# Load all models and ensemble by weighted average (weight = OOF AUC)
# ================================================================
all_preds = []
weights   = []

for model_name, score in model_scores.items():
    try:
        uri  = f'models:/{model_name}/latest'
        pipe = mlflow.sklearn.load_model(uri)
        preds = pipe.predict_proba(X_test_raw)[:, 1]
        all_preds.append(preds)
        weights.append(score)
        print(f'Loaded {model_name}')
    except Exception as e:
        print(f'Skipped {model_name}: {e}')

if len(all_preds) > 1:
    weights_arr = np.array(weights)
    weights_arr = weights_arr / weights_arr.sum()  # normalize
    ensemble_preds = sum(p * w for p, w in zip(all_preds, weights_arr))
    print(f'\nEnsemble weights: {dict(zip(model_scores.keys(), weights_arr.round(3)))}')
else:
    ensemble_preds = test_preds
    print('Only one model available — using as-is')

## 5. Create Kaggle Submission Files

In [ ]:
# --- Single best model submission ---
submission_single = pd.DataFrame({
    'TransactionID': test['TransactionID'],
    'isFraud':       test_preds
})
submission_single.to_csv('submission_best_model.csv', index=False)
print('submission_best_model.csv saved!')
submission_single.head()

In [ ]:
# --- Ensemble submission ---
submission_ensemble = pd.DataFrame({
    'TransactionID': test['TransactionID'],
    'isFraud':       ensemble_preds
})
submission_ensemble.to_csv('submission_ensemble.csv', index=False)
print('submission_ensemble.csv saved!')
submission_ensemble.describe()

## 6. Log Submission to MLflow

In [ ]:
mlflow.set_experiment('Inference')

with mlflow.start_run(run_name='Final_Submission'):
    mlflow.log_param('best_model', best_model_name)
    mlflow.log_param('ensemble_models', list(model_scores.keys()))
    mlflow.log_metric('best_model_oof_auc', model_scores[best_model_name])
    mlflow.log_artifact('submission_best_model.csv')
    mlflow.log_artifact('submission_ensemble.csv')
    print('Submission logged to MLflow!')

print('\nDone! Upload submission_best_model.csv or submission_ensemble.csv to Kaggle.')